# AISEHack 2.0 — Round 3: Polymer Property Prediction

Predicts the seven polymer properties (`tg egc egb eps nc ei eea`) from PSMILES.
Scored on the **unweighted mean R² across the seven targets**, so the four
~220-row properties (`eps nc ei eea`) carry 4/7 of the score on under 4% of the
training rows. Everything here is sized around that fact.

## Pipeline

1. Canonicalise every SMILES with RDKit. **Every downstream feature is computed
   from the canonical form**, which is what makes the pipeline exactly invariant
   to how a polymer was written (verified in §Invariance below).
2. Featurise: RDKit descriptors + Morgan(r=2,3) + atom-pair + topological-torsion
   + MACCS + polymer-specific terms + SMARTS functional groups = 2978 features.
3. Base models, per property, on one shared polymer-grouped fold assignment:
   LightGBM, XGBoost, CatBoost, a multi-task neural net and a SMILES 1-D CNN.
4. Cross-fitted Ridge stack per property.
5. Two-stage physics blend on the inter-property relations, **affine-calibrated**
   rather than applied as raw identities.
6. Clip to the observed range, write `submission.csv`.

## Runtime

Measured on an 11-core laptop, per full out-of-fold pass (10 folds for the
~220-row targets, 5 for `tg`/`egc`):

| stage | local | note |
|---|---|---|
| featurisation (12,345 molecules) | 25 s | 9 processes |
| LightGBM | 343 s | |
| XGBoost | 232 s | |
| CatBoost | 486 s | the slowest booster |
| multi-task NN, per seed | 1098 s | 2 seeds here |
| SMILES CNN, per seed | ~1100 s | 2 seeds here |
| stack + physics | ~10 s | |

Kaggle CPU sessions give 4 cores, so expect roughly 2.5x these figures — on the
order of 3-4 hours, inside the 12-hour limit. On a GPU session the two neural
models are far faster and the total drops well under 2 hours. Nothing here
branches on elapsed time, so a slower machine produces the *same* result, only
later — which is what rule 7.2 requires of the pinned version.

## Compliance

Every stage runs inside this single execution. Specifically:

* **No external data.** Only the attached competition files are read.
  `DATA_DIR` is resolved from an explicit candidate list — there is no recursive
  glob over `/kaggle/input`, so no attached dataset can be picked up silently.
* **No pretrained weights and no uploaded artifacts.** Nothing is loaded that
  this run did not itself produce: nothing is deserialised from disk, no
  checkpoint is imported, no feature cache is read. Every model is trained here.
* **No wall-clock branching.** Every training loop runs a fixed number of
  folds/epochs/iterations, so the pinned version reproduces this score exactly.
* **Seeds** are set and printed below.

A self-audit at the end of the notebook re-checks these mechanically.


In [ ]:
!pip install rdkit -q


In [ ]:
import os, sys, time, math, random, json, warnings, gc, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

SEED = 42
N_FOLDS = 10
NN_SEEDS = [42, 202]

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.use_deterministic_algorithms(False)
except Exception:
    torch = None

print(f"SEED={SEED}  N_FOLDS={N_FOLDS}  NN_SEEDS={NN_SEEDS}")
print("python", sys.version.split()[0])
print("numpy", np.__version__, "pandas", pd.__version__)


## 1. Data location and seeds


In [ ]:
# Explicit candidate list. Deliberately NOT a recursive glob over /kaggle/input:
# a glob can silently adopt an attached dataset, which is rule 6.2.1.
_CANDIDATES = [
    "/kaggle/input/aisehack-2-0",
    "/kaggle/input/aisehack-2-0-polymer-property-prediction-round-3",
    "/kaggle/input/ppp-round-3",
    "data",
    ".",
]

DATA_DIR = None
for _c in _CANDIDATES:
    if os.path.isfile(os.path.join(_c, "train.csv")) and os.path.isfile(os.path.join(_c, "test.csv")):
        DATA_DIR = _c
        break
if DATA_DIR is None:
    # One shallow scan of the immediate children of /kaggle/input -- still not
    # recursive, and it only ever matches the competition's own directory.
    _root = "/kaggle/input"
    if os.path.isdir(_root):
        for _d in sorted(os.listdir(_root)):
            _p = os.path.join(_root, _d)
            if os.path.isfile(os.path.join(_p, "train.csv")):
                DATA_DIR = _p
                break
if DATA_DIR is None:
    raise FileNotFoundError("could not locate train.csv/test.csv")

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("DATA_DIR:", DATA_DIR)
print("contents:", sorted(os.listdir(DATA_DIR))[:10])

def cache_dir():
    raise RuntimeError("no disk cache in the notebook -- everything is recomputed")


## 2. Canonicalisation and polymer rewritings

Every feature downstream is computed from `canonicalize()`. That is the whole basis of the invariance guarantee proved at the end.


In [ ]:
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
RDLogger.DisableLog('rdApp.*')
from functools import lru_cache

"""Polymer SMILES (PSMILES) canonicalisation and rewriting.

Every polymer in this competition is written as a repeat unit with EXACTLY two
`*` attachment points, e.g. `*OC(=O)c1ccc(cc1)C(=O)OCC(C(C*)CCC)CCC`.

Three ways the same polymer can be written differently:

  permutational  atom ordering inside the SMILES string          -> killed by canonicalize()
  translational  the repeat unit is cut at a different bond      -> translate() builds these
  repetition     monomer vs dimer vs trimer                      -> build_oligomer() builds these

MEASURED ON THIS DATASET (train+test, 10,605 unique raw SMILES):
  * RDKit canonicalisation collapses 10,605 raw -> 8,990 distinct polymers.
  * Grouping instead by the translation-invariant macrocycle key yields only
    7 groups that contain more than one distinct canonical SMILES, and NONE of
    those 7 differ in heavy-atom count -- i.e. the dataset contains no genuine
    oligomer (monomer/dimer/trimer) duplicates at all.

  => canonicalize() is the correct dedup/group key. macrocycle_key() must NOT be
     used to merge training rows: on the 7 borderline groups it merges polymers
     that are plausibly distinct, and it buys nothing. It exists for the
     invariance AUDIT only.
"""

from functools import lru_cache

from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

STAR = "*"


# --------------------------------------------------------------------------- #
# canonical identity                                                          #
# --------------------------------------------------------------------------- #

@lru_cache(maxsize=300_000)
def canonicalize(smi: str) -> str:
    """RDKit canonical SMILES. The group key for CV and the dedup key.

    Falls back to the raw string when RDKit cannot parse (never happens on this
    dataset -- all 10,605 unique SMILES parse -- but the export must not crash
    on a malformed test row).
    """
    if not isinstance(smi, str) or not smi:
        return smi
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi
    try:
        return Chem.MolToSmiles(m, canonical=True)
    except Exception:
        return smi


canonical_key = canonicalize  # alias: same thing, clearer at call sites in cv.py


def attachment_points(mol) -> tuple[list[int], list[int]] | None:
    """(star_indices, neighbour_indices) for a 2-star PSMILES, else None."""
    stars = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
    if len(stars) != 2:
        return None
    nbrs = []
    for s in stars:
        n = [x.GetIdx() for x in mol.GetAtomWithIdx(s).GetNeighbors()]
        if len(n) != 1:
            return None
        nbrs.append(n[0])
    return stars, nbrs


@lru_cache(maxsize=300_000)
def macrocycle_key(smi: str) -> str | None:
    """Translation-invariant key: bond the two attachment points into a ring.

    AUDIT ONLY -- see the module docstring. Returns None when the construction
    is degenerate: the two neighbours are already bonded (a 2-atom backbone such
    as `*CC*`, 1,527 of 10,605 here) or both stars hang off the same atom
    (`*C(*)R`, 13 here).
    """
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    ap = attachment_points(m)
    if ap is None:
        return None
    stars, (a, b) = ap
    if a == b or m.GetBondBetweenAtoms(a, b) is not None:
        return None
    em = Chem.RWMol(m)
    em.AddBond(a, b, Chem.BondType.SINGLE)
    for s in sorted(stars, reverse=True):
        em.RemoveAtom(s)
    try:
        mm = em.GetMol()
        Chem.SanitizeMol(mm)
        return Chem.MolToSmiles(mm)
    except Exception:
        return None


# --------------------------------------------------------------------------- #
# rewritings -- used to build the invariance certificate                      #
# --------------------------------------------------------------------------- #

def randomize(smi: str, seed: int = 0) -> str:
    """Permutational rewriting: identical graph, different atom order."""
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi
    n = m.GetNumAtoms()
    try:
        order = list(range(n))
        rng = _rng(seed)
        rng.shuffle(order)
        m2 = Chem.RenumberAtoms(m, order)
        return Chem.MolToSmiles(m2, canonical=False)
    except Exception:
        return smi


def _rng(seed: int):
    import random
    return random.Random(seed)


def build_oligomer(smi: str, n: int = 2) -> str:
    """Repetition rewriting: join n copies head-to-tail into an n-mer.

    `*A*` -> `*AA*` for n=2. Returns the input unchanged if the join is not
    possible. The result is a real RDKit-constructed molecule, not string
    concatenation -- string concatenation produces invalid SMILES for anything
    with ring-closure digits or branches.
    """
    if n < 2:
        return smi
    base = Chem.MolFromSmiles(smi)
    if base is None or attachment_points(base) is None:
        return smi
    out = base
    for _ in range(n - 1):
        out = _join(out, base)
        if out is None:
            return smi
    try:
        return Chem.MolToSmiles(out)
    except Exception:
        return smi


def _join(left, right):
    """Bond left's second attachment point to right's first, dropping both stars."""
    combo = Chem.RWMol(Chem.CombineMols(left, right))
    nL = left.GetNumAtoms()
    apL, apR = attachment_points(left), attachment_points(right)
    if apL is None or apR is None:
        return None
    (lstars, lnbrs), (rstars, rnbrs) = apL, apR
    # left keeps star[0] as the new head; right keeps star[1] as the new tail.
    drop = [lstars[1], rstars[0] + nL]
    a, b = lnbrs[1], rnbrs[0] + nL
    if a == b or combo.GetBondBetweenAtoms(a, b) is not None:
        return None
    combo.AddBond(a, b, Chem.BondType.SINGLE)
    for idx in sorted(drop, reverse=True):
        combo.RemoveAtom(idx)
    try:
        m = combo.GetMol()
        Chem.SanitizeMol(m)
        return m
    except Exception:
        return None


def translate(smi: str, k: int = 1) -> str:
    """Translational rewriting: cut the repeat unit at a different backbone bond.

    The repeat unit `*A-B-C*` and `*B-C-A*` describe the same infinite polymer;
    only the choice of cut point differs. This builds an alternative cut:
    it walks the backbone path between the two attachment points in the ORIGINAL
    molecule, closes the polymer into a macrocycle, then re-opens it at the k-th
    admissible backbone bond.

    A bond is admissible if it is single and not part of a ring in the original
    molecule -- cutting a ring bond would change the chemistry, not the cut point.
    Returns the input unchanged when no alternative cut exists (~13% of this
    dataset: two-atom backbones such as `*C(C)=C(*)CCC` where the attachment
    neighbours are already bonded, and fully-aromatic backbones where every
    backbone bond is in a ring).
    """
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi
    ap = attachment_points(m)
    if ap is None:
        return smi
    stars, (a, b) = ap
    if a == b:
        return smi

    # Backbone path in the ORIGINAL molecule. In the macrocycle this path is the
    # long way round; the shortest path there is the join bond we are about to add.
    try:
        path = Chem.GetShortestPath(m, a, b)
    except Exception:
        return smi
    if not path:
        return smi

    bonds = [(path[i], path[i + 1]) for i in range(len(path) - 1)]
    admissible = [
        (x, y) for x, y in bonds
        if (bd := m.GetBondBetweenAtoms(x, y)) is not None
        and bd.GetBondType() == Chem.BondType.SINGLE
        and not bd.IsInRing()
    ]
    if not admissible:
        return smi

    em = Chem.RWMol(m)
    if m.GetBondBetweenAtoms(a, b) is None:
        em.AddBond(a, b, Chem.BondType.SINGLE)
    for s in sorted(stars, reverse=True):
        em.RemoveAtom(s)
    try:
        ring = em.GetMol()
        Chem.SanitizeMol(ring)
    except Exception:
        return smi

    def remap(i):  # indices shift down by the number of removed stars below i
        return i - sum(1 for s in stars if s < i)

    x, y = admissible[k % len(admissible)]
    x, y = remap(x), remap(y)
    em2 = Chem.RWMol(ring)
    if em2.GetBondBetweenAtoms(x, y) is None:
        return smi
    em2.RemoveBond(x, y)
    for at in (x, y):
        s = em2.AddAtom(Chem.Atom(0))
        em2.AddBond(at, s, Chem.BondType.SINGLE)
    try:
        mm = em2.GetMol()
        Chem.SanitizeMol(mm)
        return Chem.MolToSmiles(mm)
    except Exception:
        return smi


def rewritings(smi: str, n_random: int = 3) -> dict[str, str]:
    """All rewritings of one polymer, labelled by the invariance they probe."""
    out = {"canonical": canonicalize(smi)}
    for i in range(n_random):
        out[f"permutational_{i}"] = randomize(smi, seed=i)
    out["translational"] = translate(smi, k=1)
    out["dimer"] = build_oligomer(smi, 2)
    out["trimer"] = build_oligomer(smi, 3)
    return out


## 3. The competition metric


In [ ]:
"""Exact competition metric: the UNWEIGHTED mean R^2 across the 7 target_types.

The target_type values in train.csv/test.csv are LOWERCASE. Getting this wrong
does not raise -- it silently matches zero rows and returns nan for every
target, so the assertion at import time is deliberate.
"""

import numpy as np
import pandas as pd

# Ordered by train-set size. Lowercase, exactly as they appear in the CSVs.
TARGETS: list[str] = ["tg", "egc", "egb", "eps", "nc", "ei", "eea"]

# Measured on train.csv -- used for sanity checks and clipping.
TARGET_RANGE: dict[str, tuple[float, float]] = {
    "tg":  (-109.82, 495.00),
    "egc": (0.0205, 9.8627),
    "egb": (0.5068, 10.1137),
    "eps": (2.6100, 9.0900),
    "nc":  (1.5596, 2.7581),
    "ei":  (4.0261, 9.8385),
    "eea": (0.3936, 5.1438),
}

TRAIN_COUNTS = {"tg": 4143, "egc": 2028, "egb": 337, "eps": 229, "nc": 229, "ei": 222, "eea": 221}
TEST_COUNTS = {"tg": 2763, "egc": 1352, "egb": 224, "eps": 153, "nc": 153, "ei": 148, "eea": 147}
N_TEST_ROWS = 4940  # NOT 4497 -- 4497 is the count of unique raw SMILES in test.csv


def r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_tot = float(((y_true - y_true.mean()) ** 2).sum())
    if ss_tot == 0.0:
        return float("nan")
    return 1.0 - float(((y_true - y_pred) ** 2).sum()) / ss_tot


def competition_score(df: pd.DataFrame, y_col="target", pred_col="pred",
                      type_col="target_type") -> tuple[float, dict[str, float]]:
    """(mean_score, per_target). df is long format: one row per (polymer, target_type)."""
    seen = set(df[type_col].unique())
    unknown = seen - set(TARGETS)
    if unknown:
        raise ValueError(
            f"unrecognised target_type values {sorted(unknown)}. "
            f"Expected lowercase {TARGETS}. Check for a casing bug."
        )
    per: dict[str, float] = {}
    for t in TARGETS:
        m = (df[type_col] == t).values
        per[t] = r2(df.loc[m, y_col].values, df.loc[m, pred_col].values) if m.sum() else float("nan")
    vals = [v for v in per.values() if not np.isnan(v)]
    if not vals:
        raise ValueError("no target_type matched any row -- scored nothing.")
    return float(np.mean(vals)), per


def report_metric(df, **kw) -> tuple[float, dict[str, float]]:
    score, per = competition_score(df, **kw)
    for t in TARGETS:
        n = int((df["target_type"] == t).sum())
        print(f"  {t:<4} n={n:<5} R2 = {per[t]:+.4f}")
    print(f"  {'MEAN':<4} {'':<7} = {score:+.4f}")
    return score, per


## 4. Featurisation — 2978 features from the canonical SMILES


In [ ]:
from concurrent.futures import ProcessPoolExecutor

"""Molecular featurisation, keyed by CANONICAL SMILES.

Feature families and their column prefixes (the prefix is what the
explainability report groups by):

    rd_    RDKit physicochemical descriptors      217
    mfp2_  Morgan fingerprint, radius 2          1024
    mfp3_  Morgan fingerprint, radius 3           512
    ap_    atom-pair fingerprint                  512
    tt_    topological torsion fingerprint        512
    mac_   MACCS keys                             167
    po_    polymer-specific terms                   8
    grp_   SMARTS functional-group counts          26
                                                 ----
                                                 2978

Everything is computed from the canonical SMILES, which is what makes the
pipeline invariant to how a polymer was written (see src/smiles_utils).

CACHING: `featurize()` memoises to `.cache/` so local CV iterates in seconds.
That cache is LOCAL ONLY. Competition rule 6.2.4 forbids shipping cached
feature files, so the exported notebook always recomputes from scratch
(~19 s wall for all 10,605 molecules on 9 processes -- it is not a bottleneck).
"""

import json
import os
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")


MFP2_BITS, MFP3_BITS, AP_BITS, TT_BITS = 1024, 512, 512, 512

_DESC_NAMES = [n for n, _ in Descriptors._descList]
_F32MAX = float(np.finfo(np.float32).max)

GROUP_SMARTS = {
    "aromatic_6": "[a]1[a][a][a][a][a]1", "aromatic_5": "[a]1[a][a][a][a]1",
    "amide": "[NX3][CX3](=[OX1])", "ester": "[CX3](=[OX1])[OX2]",
    "ether": "[OD2]([#6])[#6]", "hydroxyl": "[OX2H]", "carbonyl": "[CX3]=[OX1]",
    "carboxyl": "[CX3](=[OX1])[OX2H1]", "sulfonyl": "[#16X4](=[OX1])(=[OX1])",
    "sulfide": "[#16X2H0]", "nitrile": "[NX1]#[CX2]", "nitro": "[$([NX3](=O)=O)]",
    "amine_1": "[NX3;H2][#6]", "amine_2": "[NX3;H1]([#6])[#6]",
    "amine_3": "[NX3]([#6])([#6])[#6]", "imide": "[NX3](C=O)C=O",
    "urethane": "[NX3][CX3](=[OX1])[OX2]", "urea": "[NX3][CX3](=[OX1])[NX3]",
    "halide_F": "[F]", "halide_Cl": "[Cl]", "halide_Br": "[Br]", "halide_I": "[I]",
    "siloxane": "[Si][OX2][Si]", "phosphate": "[PX4](=[OX1])",
    "alkene": "[CX3]=[CX3]", "alkyne": "[CX2]#[CX2]",
}
_GROUP_PATTERNS = [(k, Chem.MolFromSmarts(v)) for k, v in GROUP_SMARTS.items()]

POLYMER_TERMS = [
    "backbone_len", "heavy_atoms", "aromatic_atoms", "n_rings", "n_hetero",
    "aromatic_ratio", "hetero_ratio", "backbone_ratio",
]


def feature_names() -> list[str]:
    return (
        [f"rd_{n}" for n in _DESC_NAMES]
        + [f"mfp2_{i}" for i in range(MFP2_BITS)]
        + [f"mfp3_{i}" for i in range(MFP3_BITS)]
        + [f"ap_{i}" for i in range(AP_BITS)]
        + [f"tt_{i}" for i in range(TT_BITS)]
        + [f"mac_{i}" for i in range(167)]
        + [f"po_{n}" for n in POLYMER_TERMS]
        + [f"grp_{k}" for k in GROUP_SMARTS]
    )


N_FEATURES = len(feature_names())

_GEN = {}


def _gens():
    if not _GEN:
        _GEN["mfp2"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MFP2_BITS)
        _GEN["mfp3"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=MFP3_BITS)
        _GEN["ap"] = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=AP_BITS)
        _GEN["tt"] = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=TT_BITS)
    return _GEN


def featurize_one(smi: str) -> np.ndarray:
    """Feature vector for one SMILES. Returns zeros if RDKit cannot parse it."""
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return np.zeros(N_FEATURES, dtype=np.float32)
    g = _gens()
    parts = []

    d = Descriptors.CalcMolDescriptors(m)
    # Ipc and a few graph descriptors can exceed the float32 range; clip before the
    # cast so the value saturates instead of silently becoming inf.
    desc = np.array([d.get(k, 0.0) for k in _DESC_NAMES], dtype=np.float64)
    desc = np.nan_to_num(desc, nan=0.0, posinf=_F32MAX, neginf=-_F32MAX)
    parts.append(np.clip(desc, -_F32MAX, _F32MAX).astype(np.float32))
    for key in ("mfp2", "mfp3", "ap", "tt"):
        parts.append(g[key].GetFingerprintAsNumPy(m).astype(np.float32))
    mac = np.zeros(167, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), mac)
    parts.append(mac)

    stars = [a.GetIdx() for a in m.GetAtoms() if a.GetAtomicNum() == 0]
    backbone = -1.0
    if len(stars) == 2:
        try:
            backbone = float(len(Chem.GetShortestPath(m, stars[0], stars[1])) - 2)
        except Exception:
            backbone = -1.0
    ha = m.GetNumHeavyAtoms()
    arom = sum(1 for a in m.GetAtoms() if a.GetIsAromatic())
    rings = m.GetRingInfo().NumRings()
    het = sum(1 for a in m.GetAtoms() if a.GetAtomicNum() not in (1, 6, 0))
    parts.append(np.array([
        backbone, ha, arom, rings, het,
        arom / max(ha, 1), het / max(ha, 1), backbone / max(ha, 1),
    ], dtype=np.float32))

    parts.append(np.array(
        [len(m.GetSubstructMatches(p)) if p is not None else 0 for _, p in _GROUP_PATTERNS],
        dtype=np.float32))

    v = np.concatenate(parts)
    return np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


_MEMO: dict[str, np.ndarray] = {}
_DISK = None


def _load_disk():
    return None


def _save_disk():
    return None


def featurize(smiles, n_jobs: int | None = None, use_cache: bool = True) -> np.ndarray:
    """(len(smiles), N_FEATURES) float32. `smiles` must already be canonical.

    Memoised PER MOLECULE, not per call, so cross-validation folds -- which each
    pass a different subset -- all hit the cache after the first pass.
    Set use_cache=False to force recomputation (what the exported notebook does).
    """
    smiles = [str(s) for s in smiles]
    if not use_cache:
        uniq = list(dict.fromkeys(smiles))
        rows = _compute(uniq, n_jobs)
        idx = {s: i for i, s in enumerate(uniq)}
        return np.vstack(rows)[[idx[s] for s in smiles]]

    _load_disk()
    uniq = list(dict.fromkeys(smiles))
    missing = [s for s in uniq if s not in _MEMO]
    if missing:
        for s, v in zip(missing, _compute(missing, n_jobs)):
            _MEMO[s] = v
        try:
            _save_disk()
        except Exception:
            pass
    return np.vstack([_MEMO[s] for s in smiles])


def _compute(uniq: list[str], n_jobs: int | None):
    """Featurise a list of SMILES, in parallel when that is actually available.

    The parallel path is a convenience, not a requirement. Inside a notebook
    every function lives in `__main__`, and a spawn-based ProcessPoolExecutor
    cannot pickle it -- so this raises `PicklingError` in Jupyter on macOS and
    Windows, and only survives on Linux because fork copies the address space.
    Relying on that would make the submission notebook platform-dependent, so any
    failure falls back to a serial loop: ~200 s for all 12,345 molecules, which
    is a rounding error against the model training that follows.
    """
    n_jobs = n_jobs or max(1, (os.cpu_count() or 4) - 2)
    if n_jobs > 1 and len(uniq) > 200 and _parallel_safe():
        try:
            with ProcessPoolExecutor(max_workers=n_jobs) as ex:
                return list(ex.map(featurize_one, uniq, chunksize=32))
        except Exception as exc:      # PicklingError, BrokenProcessPool, OSError
            print(f"featurize: parallel path failed ({type(exc).__name__}); "
                  f"falling back to serial")
    return [featurize_one(s) for s in uniq]


def _parallel_safe() -> bool:
    """Whether a process pool can actually run `featurize_one`.

    With the `fork` start method (Linux, so Kaggle) the child inherits the
    address space and any function works. With `spawn` (macOS, Windows) the
    child re-imports the function by qualified name, which is impossible when it
    was defined in a notebook cell -- everything there lives in `__main__`.
    Probing that up front avoids a BrokenProcessPool and the wall of child
    tracebacks it prints before the fallback catches it.
    """
    import multiprocessing as mp
    try:
        if mp.get_start_method(allow_none=False) == "fork":
            return True
    except Exception:
        return False
    return getattr(featurize_one, "__module__", "__main__") != "__main__"


def warm_cache(*frames) -> None:
    """Featurise every molecule in the given frames up front, once."""
    allsmi = []
    for f in frames:
        allsmi.extend(f["canon"].tolist() if hasattr(f, "columns") else list(f))
    featurize(allsmi)


def family_of(col: str) -> str:
    return col.split("_", 1)[0]


In [ ]:
# The local development harness memoises features to disk so cross-validation
# iterates in seconds. Shipping or reading such a cache would be rule 6.2.4, so
# both halves are neutralised here and every feature is recomputed from scratch.
# The in-process dict memo is kept: it only avoids featurising a molecule twice
# within THIS run (train and test share 1063 polymers).
print("feature disk cache: removed at export; every feature recomputed in this run")


## 5. Load the competition data


In [ ]:
def load_train(dedupe=True):
    df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
    df["canon"] = df["smiles"].map(canonicalize)
    unknown = set(df["target_type"].unique()) - set(TARGETS)
    assert not unknown, f"unexpected target_type {unknown}"
    if dedupe:
        df = (df.groupby(["canon", "target_type"], as_index=False)
                .agg(target=("target", "mean"), smiles=("smiles", "first")))
    return df.reset_index(drop=True)

def load_test():
    df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
    df["canon"] = df["smiles"].map(canonicalize)
    return df.reset_index(drop=True)

train_df = load_train()
test_df = load_test()
print(f"train {train_df.shape}  test {test_df.shape}")
print(train_df.target_type.value_counts().reindex(TARGETS).to_string())


## 6. Folds — one polymer-grouped assignment shared by every base model

Sharing one assignment is what makes stacking valid: each base model's out-of-fold prediction for a row comes from a model that saw exactly the same training rows. Grouping on the canonical SMILES keeps a polymer, which contributes up to six rows, inside a single fold.


In [ ]:
from sklearn.model_selection import GroupKFold, KFold

"""Cross-validation splits.

MEASURED FACT that decides the design: within a single target_type there are
essentially no duplicate canonical polymers --

    tg   4143 rows / 4139 canonical   (4 duplicate rows)
    egc  2028 / 2028      egb 337 / 337     eps 229 / 229
    nc    229 /  229      ei  222 / 222     eea 221 / 221

So a plain KFold over the rows of ONE property is already leakage-safe, and it
is what the host's own baseline notebook does. Grouping only matters for a
MULTI-TASK model, where one polymer contributes several rows: 6150 polymers have
one property, 157 have two, 126 three, 100 four, 28 five, 4 six.

Use per_property_folds() for per-property models (the normal case) and
grouped_folds() for anything that trains on the full long table at once.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, KFold

N_FOLDS_DEFAULT = 10
SMALL_N = 400          # properties below this get more folds -- less noisy OOF
N_FOLDS_SMALL = 15


def n_folds_for(n_rows: int, base: int = N_FOLDS_DEFAULT) -> int:
    """More folds for the ~220-row properties: bigger training fraction per fold
    and a less noisy OOF estimate, at negligible cost when n is this small."""
    return N_FOLDS_SMALL if n_rows < SMALL_N else base


def per_property_folds(df: pd.DataFrame, target_type: str, seed: int = 42,
                       n_folds: int | None = None):
    """Yield (train_idx, valid_idx) as positions into the rows of ONE property."""
    sub = df.index[df["target_type"] == target_type]
    n = len(sub)
    k = n_folds or n_folds_for(n)
    k = max(2, min(k, n))
    for tr, va in KFold(k, shuffle=True, random_state=seed).split(np.arange(n)):
        yield tr, va


def grouped_folds(df: pd.DataFrame, n_folds: int = N_FOLDS_DEFAULT, seed: int = 42):
    """Folds over the FULL long table, grouped so one polymer never straddles a
    split. Required for multi-task models, where a polymer contributes up to six
    rows. Yields positional indices into `df`.

    sklearn >= 1.6 gives GroupKFold a real `shuffle`/`random_state`; without it
    the split is a deterministic function of group sizes and the seed does
    nothing, which silently turns a multi-seed check into the same run twice.
    """
    groups = df["canon"].values
    try:
        gk = GroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    except TypeError:  # sklearn < 1.6
        gk = GroupKFold(n_splits=n_folds)
    for tr, va in gk.split(np.arange(len(df)), groups=groups):
        yield tr, va


## 7. Co-observed partner features, with the leakage guard

The measured values of a polymer's OTHER properties. 88-99% of DFT-block test rows have their polymer present in train under some property, which makes this the strongest feature family for that block -- and the easiest place in the pipeline to leak, so the guard is proved in both directions below.


In [ ]:
"""Co-observed partner features, with the leakage guard built in.

Many polymers carry more than one measured property. When predicting `ei` for a
polymer whose `egc` and `eea` were also measured, those measured values are
legitimate inputs -- they are in train.csv, and test polymers get the same
treatment. This is the single strongest feature family for the DFT block.

It is also the easiest place in the whole pipeline to leak. Two rules, both
enforced here rather than left to the caller:

  1. The partner table is built from the TRAINING FOLD ONLY. Building it from the
     full frame leaks validation labels into training and inflates CV.
  2. When predicting property P, the column `true_P` is dropped. It is that row's
     own label. `partners_assert_no_leak()` proves both directions: that the leak is real
     if unguarded, and that the guard removes it.

Coverage is real but partial -- the fraction of each property's TEST rows whose
partners are present in train.csv is 35-62% (see src/physics). Rows without a
partner fall back to the training-fold mean, so the model must also work without.
"""

import numpy as np
import pandas as pd


# `tg` is measured on a disjoint set of polymers (only 4 of 4143 tg polymers
# carry any other property), so it is neither a useful partner nor helped by one.
DFT_PROPS = ["egc", "egb", "eps", "nc", "ei", "eea"]


def partners_build(train_fold: pd.DataFrame, frames: list[pd.DataFrame]):
    """Return (list_of_feature_frames, column_names).

    `train_fold` supplies the labels. `frames` are the frames to featurise
    (typically [train_fold, valid_fold] or [train, test]).
    """
    wide = train_fold.pivot_table(index="canon", columns="target_type",
                                  values="target", aggfunc="mean")
    cols, fills = [], {}
    for p in DFT_PROPS:
        if p not in wide.columns:
            continue
        cols.append(f"true_{p}")
        fills[f"true_{p}"] = float(wide[p].mean())

    out = []
    for df in frames:
        block = pd.DataFrame(index=range(len(df)))
        for p in DFT_PROPS:
            if f"true_{p}" not in cols:
                continue
            v = wide[p].reindex(df["canon"].values).values
            c = f"true_{p}"
            block[c] = np.where(np.isfinite(v), v, fills[c])
            block[f"has_{p}"] = np.isfinite(v).astype(np.float32)
        block["n_partners"] = block[[f"has_{p}" for p in DFT_PROPS
                                     if f"has_{p}" in block]].sum(axis=1)
        out.append(block.astype(np.float32))
    names = list(out[0].columns) if out else []
    return out, names


def partners_drop_leaky(block: pd.DataFrame, target_type: str) -> pd.DataFrame:
    """Remove the columns that encode this row's own label."""
    banned = {f"true_{target_type}", f"has_{target_type}"}
    return block[[c for c in block.columns if c not in banned]]


def partners_assert_no_leak(train: pd.DataFrame) -> None:
    """Prove the leak exists, then prove partners_drop_leaky() removes it.

    A guard that passes without a demonstrable leak proves nothing, so this
    checks both directions and raises on either failure.
    """
    blocks, _ = partners_build(train, [train])
    block = blocks[0]
    for p in DFT_PROPS:
        m = (train["target_type"] == p).values
        if m.sum() == 0 or f"true_{p}" not in block:
            continue
        got = block.loc[m, f"true_{p}"].values
        want = train.loc[m, "target"].values
        if not np.allclose(got, want, atol=1e-6):
            raise AssertionError(
                f"true_{p} does not reproduce the {p} target -- partner partners_build is wrong")
        if f"true_{p}" in partners_drop_leaky(block.loc[m], p).columns:
            raise AssertionError(f"partners_drop_leaky failed to remove true_{p}")
    print(f"partner leakage guard OK ({len(DFT_PROPS)} properties: leak demonstrated, guard removes it)")


In [ ]:
partners_assert_no_leak(train_df)


## 8. Physics relations between the DFT properties

Measured on co-observed training pairs: the raw identity `eps = nc**2` scores R² 0.336, while the same expression affine-calibrated scores 0.855. `nc` goes 0.171 → 0.837 and `egb` 0.892 → 0.928. Every relation here is fitted, never applied raw.


In [ ]:
from dataclasses import dataclass, field

"""Physics relations between the DFT properties.

The four relations, MEASURED on co-observed train pairs. `raw` is the textbook
identity; `fitted` is the same expression passed through a 1-D least-squares
calibration a*x + b estimated on the training co-observations:

    target  expression        n     raw R2    fitted R2   a       b
    ei      egc + eea         59    0.9629    0.9650      1.005    0.013
    eea     ei  - egc         59    0.9710    0.9727      1.004   -0.054
    egb     egc              175    0.8922    0.9282      1.159   -1.044
    eps     nc ** 2          134    0.3364    0.8553      1.040    0.615
    nc      sqrt(eps)        134    0.1708    0.8370      0.887    0.050

Read the eps and nc rows carefully. The Maxwell relation eps = n^2 holds for the
optical dielectric constant; the measured static eps sits well above it, so the
RAW identity is badly biased (R2 0.34) while the SAME expression with a fitted
offset is worth 0.86. Both shipped notebooks apply the raw form. Always fit.

Coverage on test -- the fraction of a property's test rows whose partner values
are present in train.csv, i.e. the rows this can actually touch with TRUE inputs:

    ei 55/148 (37%)   eea 51/147 (35%)   egb 124/224 (55%)
    eps 95/153 (62%)  nc 95/153 (62%)
"""

from dataclasses import dataclass, field

import numpy as np
import pandas as pd


@dataclass
class Relation:
    target: str
    sources: list[str]
    expr: callable
    a: float = 1.0
    b: float = 0.0
    n_fit: int = 0
    raw_r2: float = float("nan")
    fitted_r2: float = float("nan")

    def apply(self, src: np.ndarray) -> np.ndarray:
        return self.a * self.expr(src) + self.b


RELATIONS: dict[str, tuple[list[str], callable]] = {
    "ei":  (["egc", "eea"], lambda d: d[:, 0] + d[:, 1]),
    "eea": (["ei", "egc"],  lambda d: d[:, 0] - d[:, 1]),
    "egb": (["egc"],        lambda d: d[:, 0]),
    "eps": (["nc"],         lambda d: d[:, 0] ** 2),
    "nc":  (["eps"],        lambda d: np.sqrt(np.clip(d[:, 0], 0, None))),
}


def wide_table(train: pd.DataFrame) -> pd.DataFrame:
    """canon -> one column per target_type (NaN where not measured)."""
    return train.pivot_table(index="canon", columns="target_type",
                             values="target", aggfunc="mean")


def _r2(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float)
    ss = ((y - y.mean()) ** 2).sum()
    return float("nan") if ss == 0 else 1.0 - float(((y - p) ** 2).sum()) / ss


def fit_relations(train: pd.DataFrame, min_n: int = 20) -> dict[str, Relation]:
    """Calibrate every relation on the co-observed rows of `train` ONLY.

    Must be called with the CV training fold, never the full frame, or the
    calibration leaks validation labels.
    """
    w = wide_table(train)
    out: dict[str, Relation] = {}
    for tgt, (srcs, expr) in RELATIONS.items():
        if tgt not in w.columns or any(s not in w.columns for s in srcs):
            continue
        d = w.dropna(subset=[tgt] + srcs)
        if len(d) < min_n:
            continue
        x = expr(d[srcs].values)
        y = d[tgt].values
        if np.std(x) < 1e-12:
            continue
        a, b = np.polyfit(x, y, 1)
        out[tgt] = Relation(tgt, srcs, expr, float(a), float(b), len(d),
                            _r2(y, x), _r2(y, a * x + b))
    return out


def partner_estimate(df: pd.DataFrame, rels: dict[str, Relation],
                     lookup: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Physics estimate per row, using TRUE partner labels from `lookup`.

    Returns (estimate, covered_mask). Rows whose partners are absent get NaN.
    """
    est = np.full(len(df), np.nan)
    for tgt, rel in rels.items():
        m = (df["target_type"] == tgt).values
        if not m.any():
            continue
        idx = df.loc[m, "canon"]
        cols = [c for c in rel.sources if c in lookup.columns]
        if len(cols) != len(rel.sources):
            continue
        src = lookup.reindex(idx)[rel.sources].values
        ok = np.isfinite(src).all(axis=1)
        vals = np.full(len(src), np.nan)
        if ok.any():
            vals[ok] = rel.apply(src[ok])
        est[m] = vals
    return est, np.isfinite(est)


def blend(model_pred: np.ndarray, phys_est: np.ndarray, weight: float) -> np.ndarray:
    """Convex blend where the physics estimate exists, model prediction elsewhere."""
    out = np.asarray(model_pred, float).copy()
    m = np.isfinite(phys_est)
    out[m] = (1.0 - weight) * out[m] + weight * np.asarray(phys_est, float)[m]
    return out


def tune_weight(y_true, model_pred, phys_est, grid=None) -> tuple[float, float]:
    """Pick the blend weight that maximises R2 on the rows physics covers.

    Tune on OOF predictions, per target. Returns (best_weight, best_r2).
    """
    grid = grid if grid is not None else np.linspace(0.0, 1.0, 21)
    m = np.isfinite(phys_est)
    if m.sum() < 10:
        return 0.0, _r2(y_true, model_pred)
    best_w, best = 0.0, _r2(y_true, model_pred)
    for w in grid:
        s = _r2(y_true, blend(model_pred, phys_est, w))
        if s > best:
            best, best_w = s, float(w)
    return best_w, best


def report_relations(rels: dict[str, Relation]) -> None:
    print(f"{'target':<7}{'sources':<14}{'n':>5}{'raw R2':>10}{'fitted R2':>12}{'a':>8}{'b':>8}")
    for t, r in rels.items():
        print(f"{t:<7}{'+'.join(r.sources):<14}{r.n_fit:>5}{r.raw_r2:>10.4f}"
              f"{r.fitted_r2:>12.4f}{r.a:>8.3f}{r.b:>8.3f}")


## 9. Base models


In [ ]:
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

def params_for(n: int, seed: int) -> dict:
    big = n > 1000
    return dict(
        objective="regression", metric="rmse", verbosity=-1,
        n_estimators=1200 if big else 900,
        learning_rate=0.03,
        num_leaves=31 if big else 15,
        max_depth=7 if big else 5,
        min_child_samples=10 if big else 5,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.6,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=seed, n_jobs=-1,
    )


_lgbm_params_for = params_for


"""Per-property gradient-boosting estimators: LightGBM, XGBoost, CatBoost.

Uniform factory so the CV driver can treat all three identically:

    pass
    est = trees.make('lgbm', 'eps', n_rows=229, seed=42)
    est.fit(X_tr, y_tr).predict(X_va)

WHY CAPACITY IS A FUNCTION OF n_rows
------------------------------------
Per-target train sizes span a factor of 19 (tg 4139 ... eea 221) and every
target is worth the same 1/7 of the metric. One hyper-parameter set sized for
`tg` over-fits the four ~220-row DFT properties, which together are 4/7 of the
score. `_tier()` buckets n into small/mid/large and every builder switches on
it: fewer leaves, shallower trees, larger minimum leaf occupancy and stronger
L2 as n shrinks.

The lgbm branch is anchored on `src.configs.lgbm.params_for` so this module and
the reference config cannot drift apart; xgb and cb are capacity-matched
analogues (lgbm `num_leaves=31` ~ a depth-5 symmetric tree, `num_leaves=15` ~
depth 4).

COMPETITION-RULE NOTES
----------------------
* Trains from scratch on whatever it is handed. No checkpoints, no pickles, no
  downloads, no reading or writing files -- CatBoost's `allow_writing_files` is
  forced off so it cannot drop a `catboost_info/` directory.
* No wall-clock branching anywhere. Iteration counts are fixed constants.
* No early stopping inside `.fit()`: the driver owns the folds, and an
  early-stopping set carved out here would leak the fold's own validation rows.
* Deterministic for a fixed (seed, n_jobs). CatBoost and LightGBM both partition
  work by thread, so reproducing a score requires pinning `n_jobs` too --
  `DEFAULT_N_JOBS` is derived from `os.cpu_count()`, so pass it explicitly if
  local and Kaggle core counts differ and you need bit-identical output.
"""

import os

import numpy as np


_NAME_TREES = "trees"
ESTIMATORS = ["lgbm", "xgb", "cb"]

# Kaggle CPU sessions give 4 cores, this machine has 11. Derived, never hardcoded.
DEFAULT_N_JOBS = max(1, os.cpu_count() or 4)

SMALL_N = 400     # eps/nc/ei/eea (~220) and egb (337) land here
MID_N = 1500      # egc (2028) is 'large', nothing currently lands in 'mid'


def _tier(n_rows: int) -> str:
    if n_rows < SMALL_N:
        return "small"
    if n_rows < MID_N:
        return "mid"
    return "large"


def _jobs(n_jobs: int | None) -> int:
    return DEFAULT_N_JOBS if n_jobs is None else max(1, int(n_jobs))


# --------------------------------------------------------------------------- params

def lgbm_params(n_rows: int, seed: int, n_jobs: int | None = None) -> dict:
    """Reference `params_for` plus a middle tier and an explicit thread count."""
    p = dict(_lgbm_params_for(int(n_rows), int(seed)))
    if _tier(n_rows) == "mid":
        p.update(n_estimators=1100, num_leaves=23, max_depth=6, min_child_samples=8)
    p["n_jobs"] = _jobs(n_jobs)
    p["random_state"] = int(seed)
    return p


def xgb_params(n_rows: int, seed: int, n_jobs: int | None = None) -> dict:
    tier = _tier(n_rows)
    depth = {"small": 4, "mid": 5, "large": 6}[tier]
    return dict(
        n_estimators={"small": 900, "mid": 1100, "large": 1200}[tier],
        learning_rate=0.03,
        max_depth=depth,
        min_child_weight={"small": 5.0, "mid": 4.0, "large": 3.0}[tier],
        subsample=0.8,
        colsample_bytree=0.6,
        reg_alpha=0.1,
        reg_lambda={"small": 3.0, "mid": 2.0, "large": 1.0}[tier],
        tree_method="hist",
        max_bin=256,
        objective="reg:squarederror",
        random_state=int(seed),
        n_jobs=_jobs(n_jobs),
        verbosity=0,
    )


def cb_params(n_rows: int, seed: int, n_jobs: int | None = None) -> dict:
    """CatBoost's symmetric trees cost ~2^depth leaves, so depth moves one step
    below the xgb depth-wise analogue. `rsm` is CatBoost's colsample; with 2978
    columns it is also the main runtime lever."""
    tier = _tier(n_rows)
    return dict(
        iterations={"small": 900, "mid": 1100, "large": 1200}[tier],
        learning_rate=0.04,
        depth={"small": 4, "mid": 5, "large": 6}[tier],
        l2_leaf_reg={"small": 6.0, "mid": 4.0, "large": 3.0}[tier],
        min_data_in_leaf={"small": 5, "mid": 8, "large": 10}[tier],
        rsm=0.3,
        bootstrap_type="Bernoulli",
        subsample=0.8,
        border_count={"small": 64, "mid": 96, "large": 128}[tier],
        loss_function="RMSE",
        random_seed=int(seed),
        thread_count=_jobs(n_jobs),
        allow_writing_files=False,   # otherwise it drops catboost_info/ on disk
        # `verbose` and `logging_level` are mutually exclusive in catboost 1.2 --
        # setting both raises. 'Silent' is the one that also mutes the fit banner.
        logging_level="Silent",
    )


# --------------------------------------------------------------------------- wrapper

class _Boosted:
    """Uniform `.fit(X, y, smiles=None)` / `.predict(X, smiles=None)` shim.

    `smiles` is accepted and ignored: these are pure tabular learners, but the
    driver calls every estimator family through one signature and the SMILES
    models in sibling modules need it.
    """

    __slots__ = ("kind", "target_type", "n_rows", "seed", "params", "model", "_const")

    def __init__(self, kind, target_type, n_rows, seed, params, model):
        self.kind = kind
        self.target_type = target_type
        self.n_rows = int(n_rows)
        self.seed = int(seed)
        self.params = params
        self.model = model
        self._const = None

    def fit(self, X, y, smiles=None):
        X = np.ascontiguousarray(np.asarray(X, dtype=np.float32))
        y = np.asarray(y, dtype=np.float64).ravel()
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"X has {X.shape[0]} rows, y has {y.shape[0]}")
        # Degenerate folds: a constant or near-empty target makes every booster
        # either warn loudly or fail outright. Predict the mean instead.
        if len(y) < 5 or float(np.ptp(y)) == 0.0:
            self._const = float(y.mean()) if len(y) else 0.0
            return self
        self._const = None
        self.model.fit(X, y)
        return self

    def predict(self, X, smiles=None):
        X = np.ascontiguousarray(np.asarray(X, dtype=np.float32))
        if self._const is not None:
            return np.full(X.shape[0], self._const, dtype=np.float64)
        return np.asarray(self.model.predict(X), dtype=np.float64).ravel()

    def feature_importance(self):
        """Gain-style importance, aligned with `src.features.feature_names()`.
        Returns None before `.fit()` or on a degenerate constant fit."""
        if self._const is not None:
            return None
        if self.kind == "lgbm":
            return np.asarray(self.model.booster_.feature_importance("gain"), dtype=np.float64)
        if self.kind == "xgb":
            return np.asarray(self.model.feature_importances_, dtype=np.float64)
        return np.asarray(self.model.get_feature_importance(), dtype=np.float64)

    def __repr__(self):
        return (f"<{self.kind} target={self.target_type} n={self.n_rows} "
                f"tier={_tier(self.n_rows)} seed={self.seed}>")


def make(kind: str, target_type: str, n_rows: int, seed: int, *, n_jobs: int | None = None):
    """Build one unfitted per-property booster.

    kind         one of ESTIMATORS
    target_type  the property this estimator serves; recorded, not branched on,
                 so capacity is decided by data size rather than by name
    n_rows       rows this estimator will be FIT on (i.e. the training fold, not
                 the whole target) -- that is what capacity must track
    seed         propagated to every source of randomness
    n_jobs       threads; defaults to DEFAULT_N_JOBS
    """
    kind = str(kind).lower()
    n_rows = int(n_rows)
    seed = int(seed)

    if kind == "lgbm":
        import lightgbm as lgb
        p = lgbm_params(n_rows, seed, n_jobs)
        model = lgb.LGBMRegressor(**p)
    elif kind == "xgb":
        from xgboost import XGBRegressor
        p = xgb_params(n_rows, seed, n_jobs)
        model = XGBRegressor(**p)
    elif kind == "cb":
        from catboost import CatBoostRegressor
        p = cb_params(n_rows, seed, n_jobs)
        model = CatBoostRegressor(**p)
    else:
        raise ValueError(f"unknown kind {kind!r}; expected one of {ESTIMATORS}")

    return _Boosted(kind, target_type, n_rows, seed, p, model)


### Multi-task neural network

One shared trunk over the feature matrix with seven per-property heads. `y` is standardised **per target** inside each fold — without that, `tg` (range 495) dominates a shared loss and `nc` (range 2.76) learns nothing.


In [ ]:
"""Multi-task neural network: one shared trunk over the 2978-dim feature matrix,
seven per-property heads.

Why multi-task here
-------------------
The four small properties (eps/nc/ei/eea, ~220 rows each) are 4/7 of the score and
have far too few rows to support an independent deep model. But 5920 distinct
polymers carry *some* label, and the polymers that carry eps also tend to carry
nc/ei/eea/egc -- so a shared trunk trained on the whole long table sees 27x more
molecules than an eea-only model does, and only the 8k-parameter eea head has to
be learned from 221 rows.

Two things make or break this model, and both are handled per fold, fit on the
fold's TRAINING rows only:

1. y is standardised PER TARGET. tg reaches 495 and nc 2.76; a shared MSE on raw
   targets is ~99.9% tg and every other head collapses to its mean.
2. X is signed-log1p compressed then standardised. Raw RDKit descriptors reach
   3.4e38 (Ipc and friends), which is an instant NaN through a BatchNorm.

Rows are collapsed to one row per polymer with a (n_mol, 7) label matrix and a
boolean observation mask, so one forward pass updates every head that polymer has
a label for. The loss is a masked MSE, averaged per target then combined with
fixed weights, so tg's 4139 rows do not drown eea's 221.

Competition compliance: trains from scratch, fixed epoch count with a
deterministic cosine schedule (no wall-clock branching, no early stopping against
the validation fold), no file I/O, no pretrained anything.
"""

import numpy as np
import torch
import torch.nn as nn

_NAME_MTNN = "mtnn"
DESCRIPTION = "multi-task MLP, shared trunk + 7 per-property heads, masked MSE"

try:  # keep the flattened-notebook copy working standalone
    pass
except Exception:  # pragma: no cover
    TARGETS = ["tg", "egc", "egb", "eps", "nc", "ei", "eea"]

# ---------------------------------------------------------------- hyperparams
HP = dict(
    trunk=(1024, 512, 256, 128),
    head_dim_big=64,        # head width for targets with > SMALL_N training rows
    head_dim_small=32,      # head width for the ~220-row targets
    small_n=1000,
    dropout=0.30,
    head_dropout=0.10,
    epochs=140,
    batch_size=512,
    lr=2.0e-3,
    weight_decay=1.0e-4,
    warmup_frac=0.08,
    final_lr_frac=0.02,
    loss_alpha=0.25,        # per-target loss weight = n_t ** alpha (0 = balanced,
                            # 1 = pooled MSE dominated by tg)
    snapshots=3,            # average predictions from the last k epochs
    snapshot_stride=6,
    clip_z=10.0,            # clip standardised features
    min_std=1e-6,
)


# ------------------------------------------------------------------- network
class _MultiTaskNet(nn.Module):
    def __init__(self, n_in: int, head_dims: list[int], hp: dict):
        super().__init__()
        layers: list[nn.Module] = []
        p = n_in
        for width in hp["trunk"]:
            layers += [nn.Linear(p, width), nn.BatchNorm1d(width), nn.ReLU(),
                       nn.Dropout(hp["dropout"])]
            p = width
        self.trunk = nn.Sequential(*layers)
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(p, h), nn.ReLU(), nn.Dropout(hp["head_dropout"]),
                          nn.Linear(h, 1))
            for h in head_dims
        ])

    def forward(self, x):
        h = self.trunk(x)
        return torch.cat([head(h) for head in self.heads], dim=1)


# --------------------------------------------------------------- preparation
def _signed_log1p(X: np.ndarray) -> np.ndarray:
    """Tame the dynamic range without fitting anything (so it cannot leak).

    Monotone and sign-preserving: 3.4e38 -> 88.7, and the 2727 binary fingerprint
    bits are merely rescaled 0/1 -> 0/0.693.
    """
    Z = np.asarray(X, dtype=np.float64)
    Z = np.nan_to_num(Z, nan=0.0, posinf=np.finfo(np.float32).max,
                      neginf=-np.finfo(np.float32).max)
    return (np.sign(Z) * np.log1p(np.abs(Z))).astype(np.float32)


def _collapse(df, X, tgt_index: dict[str, int], fold_id=None):
    """Long rows -> one row per polymer, plus (n_mol, 7) label matrix and mask.

    Returns mol_of_row (row -> molecule slot), Xm, Y, M, mol_fold.
    """
    canon = df["canon"].to_numpy()
    slot: dict[str, int] = {}
    mol_of_row = np.empty(len(df), dtype=np.int64)
    first_row: list[int] = []
    for i, c in enumerate(canon):
        j = slot.get(c)
        if j is None:
            j = len(first_row)
            slot[c] = j
            first_row.append(i)
        mol_of_row[i] = j
    first_row_arr = np.asarray(first_row, dtype=np.int64)
    n_mol = len(first_row_arr)

    Xm = np.ascontiguousarray(X[first_row_arr])

    Y = np.zeros((n_mol, len(tgt_index)), dtype=np.float32)
    M = np.zeros((n_mol, len(tgt_index)), dtype=bool)
    if "target" in df.columns:
        tt = df["target_type"].to_numpy()
        yv = df["target"].to_numpy(dtype=np.float64)
        cnt = np.zeros_like(Y)
        for r in range(len(df)):
            c = tgt_index[tt[r]]
            m = mol_of_row[r]
            Y[m, c] += yv[r]
            cnt[m, c] += 1.0
            M[m, c] = True
        Y = np.divide(Y, np.where(cnt > 0, cnt, 1.0)).astype(np.float32)

    # Fold assignment is per (molecule, target) CELL, not per molecule.
    # This project uses per-property folds, so one polymer legitimately has its
    # tg row in fold 3 and its egc row in fold 7. That is not a bug to reject --
    # it is what mirrors inference, where a test polymer's other measured
    # properties are in train and the network has already seen its structure.
    # A molecule therefore trains on its non-held-out targets while being
    # predicted for its held-out ones.
    cell_fold = None
    if fold_id is not None:
        fid = np.asarray(fold_id).astype(np.int64)
        cell_fold = np.full((n_mol, len(tgt_index)), -1, dtype=np.int64)
        tt_arr = df["target_type"].to_numpy()
        for r in range(len(df)):
            cell_fold[mol_of_row[r], tgt_index[tt_arr[r]]] = fid[r]
    return mol_of_row, Xm, Y, M, cell_fold


# ------------------------------------------------------------------ training
def _train_one(Xtr, Ytr, Mtr, head_dims, w_task, hp, seed, device, predict_sets):
    """Train one network and return averaged snapshot predictions for each
    matrix in `predict_sets` (standardised-y space)."""
    torch.manual_seed(seed)
    if device == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    dev = torch.device(device)
    net = _MultiTaskNet(Xtr.shape[1], head_dims, hp).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=hp["lr"],
                            weight_decay=hp["weight_decay"])

    xt = torch.from_numpy(Xtr).to(dev)
    yt = torch.from_numpy(Ytr).to(dev)
    mt = torch.from_numpy(Mtr.astype(np.float32)).to(dev)
    wt = torch.from_numpy(w_task.astype(np.float32)).to(dev)

    n = xt.shape[0]
    bs = min(hp["batch_size"], n)
    steps_per_epoch = max(1, int(np.ceil(n / bs)))
    total_steps = hp["epochs"] * steps_per_epoch
    warm = max(1, int(hp["warmup_frac"] * total_steps))
    fl = hp["final_lr_frac"]

    def lr_at(step: int) -> float:
        if step < warm:
            return hp["lr"] * (step + 1) / warm
        prog = (step - warm) / max(1, total_steps - warm)
        return hp["lr"] * (fl + (1.0 - fl) * 0.5 * (1.0 + np.cos(np.pi * prog)))

    rng = np.random.default_rng(seed)
    snap_epochs = {hp["epochs"] - 1 - k * hp["snapshot_stride"]
                   for k in range(hp["snapshots"])}
    snap_epochs = {e for e in snap_epochs if e >= 0}
    acc = [np.zeros((len(P), len(head_dims)), dtype=np.float64) for P in predict_sets]
    n_snap = 0

    step = 0
    for ep in range(hp["epochs"]):
        net.train()
        order = rng.permutation(n)
        for s in range(steps_per_epoch):
            idx = torch.from_numpy(order[s * bs:(s + 1) * bs]).to(dev)
            if idx.numel() < 2:      # BatchNorm needs >= 2 rows
                continue
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            step += 1
            xb, yb, mb = xt[idx], yt[idx], mt[idx]
            pred = net(xb)
            se = (pred - yb) ** 2 * mb
            cnt = mb.sum(0)
            per_task = se.sum(0) / cnt.clamp(min=1.0)
            present = (cnt > 0).float()
            w = wt * present
            loss = (w * per_task).sum() / w.sum().clamp(min=1e-8)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()

        if ep in snap_epochs:
            net.eval()
            with torch.no_grad():
                for k, P in enumerate(predict_sets):
                    if len(P) == 0:
                        continue
                    pt = torch.from_numpy(P).to(dev)
                    out = []
                    for b in range(0, len(P), 4096):
                        out.append(net(pt[b:b + 4096]).cpu().numpy())
                    acc[k] += np.vstack(out).astype(np.float64)
            n_snap += 1
    return [a / max(1, n_snap) for a in acc]


# -------------------------------------------------------------------- driver
def mtnn_oof_and_test(train_df, X_tr, test_df, X_te, fold_id, seeds,
                 n_jobs=None, device=None, hp: dict | None = None):
    """Out-of-fold and test predictions from the multi-task network.

    train_df : long format, columns target_type / target / canon
    X_tr     : float32 (len(train_df), n_feat), row-aligned with train_df
    test_df  : columns target_type / canon
    X_te     : float32 (len(test_df), n_feat), row-aligned with test_df
    fold_id  : int array (len(train_df),), grouped by polymer
    seeds    : list[int]; trained once per seed, predictions averaged
    returns  : (oof, test_pred) float64 1-D, len(train_df) / len(test_df)
    """
    hp = {**HP, **(hp or {})}
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if n_jobs:
        torch.set_num_threads(int(n_jobs))
    seeds = list(seeds) if seeds is not None else [42]

    tgt_index = {t: i for i, t in enumerate(TARGETS)}
    n_t = len(TARGETS)
    unknown = (set(train_df["target_type"]) | set(test_df["target_type"])) - set(TARGETS)
    if unknown:
        raise ValueError(f"mtnn: unrecognised target_type {sorted(unknown)}")

    X_tr = np.asarray(X_tr, dtype=np.float32)
    X_te = np.asarray(X_te, dtype=np.float32)
    if X_tr.shape[1] != X_te.shape[1]:
        raise ValueError("mtnn: train/test feature widths differ")

    Ztr_full = _signed_log1p(X_tr)
    Zte_full = _signed_log1p(X_te)

    mol_of_row, Zm, Y, M, cell_fold = _collapse(train_df, Ztr_full, tgt_index, fold_id)
    te_mol_of_row, Zte, _, _, _ = _collapse(test_df, Zte_full, tgt_index)
    te_col = test_df["target_type"].map(tgt_index).to_numpy(dtype=np.int64)
    tr_col = train_df["target_type"].map(tgt_index).to_numpy(dtype=np.int64)

    counts = M.sum(0).astype(np.float64)
    head_dims = [hp["head_dim_big"] if c > hp["small_n"] else hp["head_dim_small"]
                 for c in counts]
    w_task = np.where(counts > 0, np.maximum(counts, 1.0) ** hp["loss_alpha"], 0.0)

    folds = np.unique(cell_fold[cell_fold >= 0])
    oof_mol = np.zeros((len(Zm), n_t), dtype=np.float64)
    oof_hits = np.zeros((len(Zm), n_t), dtype=np.int64)
    test_mol = np.zeros((len(Zte), n_t), dtype=np.float64)
    test_hits = 0

    for f in folds:
        Mva = M & (cell_fold == f)          # cells held out this fold
        Mtr = M & (cell_fold != f)          # cells trainable this fold
        tr = Mtr.any(axis=1)                # molecules with something to learn from
        va = Mva.any(axis=1)                # molecules with something to predict
        if tr.sum() < 10 or va.sum() == 0:
            continue

        # --- feature scaler: fit on this fold's TRAINING molecules only
        mu = Zm[tr].mean(0)
        sd = Zm[tr].std(0)
        keep = sd > hp["min_std"]
        mu_k, sd_k = mu[keep], sd[keep]
        cz = hp["clip_z"]
        Xtr = np.clip((Zm[tr][:, keep] - mu_k) / sd_k, -cz, cz).astype(np.float32)
        Xva = np.clip((Zm[va][:, keep] - mu_k) / sd_k, -cz, cz).astype(np.float32)
        Xte_f = np.clip((Zte[:, keep] - mu_k) / sd_k, -cz, cz).astype(np.float32)

        # --- y scaler: per target, fold-training cells of that target only
        Mtr_sub = Mtr[tr]
        Ytr_raw = Y[tr]
        y_mu = np.zeros(n_t, dtype=np.float64)
        y_sd = np.ones(n_t, dtype=np.float64)
        Ytr = np.zeros_like(Ytr_raw)
        for c in range(n_t):
            m = Mtr_sub[:, c]
            if m.sum() < 2:
                continue
            vals = Ytr_raw[m, c].astype(np.float64)
            y_mu[c] = vals.mean()
            s = vals.std()
            y_sd[c] = s if s > 1e-9 else 1.0
            Ytr[m, c] = ((vals - y_mu[c]) / y_sd[c]).astype(np.float32)

        for seed in seeds:
            pv, pt = _train_one(Xtr, Ytr, Mtr_sub, head_dims, w_task, hp,
                                int(seed) + 1009 * int(f), device, [Xva, Xte_f])
            pv = pv * y_sd + y_mu
            pt = pt * y_sd + y_mu
            # accumulate ONLY the cells actually held out this fold
            oof_mol[va] += pv * Mva[va]
            test_mol += pt
            test_hits += 1
        oof_hits[va] += Mva[va] * len(seeds)

    seen = oof_hits[mol_of_row, tr_col]
    if (seen == 0).any():
        raise ValueError(
            f"mtnn: {int((seen == 0).sum())} training rows were never held out")
    oof_mol = np.divide(oof_mol, np.where(oof_hits > 0, oof_hits, 1))
    test_mol /= max(1, test_hits)

    oof = oof_mol[mol_of_row, tr_col].astype(np.float64)
    test_pred = test_mol[te_mol_of_row, te_col].astype(np.float64)

    if not np.isfinite(oof).all():
        raise ValueError("mtnn: non-finite values in oof")
    if not np.isfinite(test_pred).all():
        raise ValueError("mtnn: non-finite values in test_pred")
    assert oof.shape == (len(train_df),) and test_pred.shape == (len(test_df),)
    return oof, test_pred


## 10. Out-of-fold engine, cross-fitted stack, two-stage physics

The stacker, the physics blend weight and the physics calibration are all cross-fitted over the fold assignment. Fitting them on the same out-of-fold predictions they are then scored against is what produces an OOF number that does not survive the leaderboard.


In [ ]:
from sklearn.linear_model import Ridge

"""Out-of-fold engine: base models -> cross-fitted stack -> two-stage physics.

The whole pipeline in one place, so the number this prints is the number the
notebook reproduces.

Three things here are deliberately cross-fitted rather than fitted on the full
OOF. Fitting a stacker or a blend weight on the same out-of-fold predictions you
then score is the classic way to get an OOF number that does not survive contact
with the leaderboard, and it is the leading explanation for the team's
0.903 OOF -> 0.883 public LB gap:

  1. the Ridge stacker            cross-fitted over `fold_id`
  2. the physics blend weight     cross-fitted over `fold_id`
  3. the physics calibration a,b  fitted on training-fold rows only

FOLDS. One global fold assignment, grouped on canonical SMILES, shared by every
base model. Grouping is only strictly required for the multi-task models (a
polymer contributes up to six rows there), but sharing one assignment is what
makes stacking valid: every base model's OOF for a given row comes from a model
that saw exactly the same training rows.
"""

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge


N_FOLDS = 10
SEED = 42
STACK_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]


# --------------------------------------------------------------------------- #
# folds                                                                        #
# --------------------------------------------------------------------------- #

def folds_for_target(n_rows: int, base: int = N_FOLDS) -> int:
    """Fold count scaled to sample size, in the direction runtime wants.

    tg (4139 rows) and egc (2028) sit at R2 ~0.91 and are the two expensive
    targets: extra folds buy almost nothing there and dominate wall time. The
    four ~220-row targets are noisy and cheap, so they get more folds -- a bigger
    training fraction per fold and a less noisy OOF estimate, for free.
    """
    if n_rows > 1500:
        return max(2, min(base, 5))
    if n_rows >= 400:
        return base
    return max(base, 15)


def build_fold_id(train_df: pd.DataFrame, n_folds: int = N_FOLDS,
                  seed: int = SEED, scheme: str = "per_property") -> np.ndarray:
    """Fold assignment for every training row.

    scheme="per_property" (the default, and the correct one here): KFold within
    each target_type independently. Leakage-safe, because within a target_type
    there are essentially no duplicate canonical polymers.

    scheme="grouped": one polymer never straddles a fold.

    WHY per_property AND NOT grouped, measured:

        partner availability for validation rows, grouped folds ... 0%
        partner availability for test rows, actual test set ....... 88-99%

    A polymer held out entirely loses the other properties measured on it, so
    grouped CV evaluates a model that has been denied a signal present for
    almost every real test row in the DFT block. It is not "conservative", it
    measures a different problem. Under per_property folds a held-out ei row
    keeps its polymer's egc and eea rows in training -- exactly the situation at
    test time, where 98% of ei test polymers appear in train under some property.

    The same argument covers the multi-task models: at test time the network has
    already seen a test polymer's structure whenever that polymer carries some
    other measured property, so per_property folds mirror inference there too.
    """
    fold_id = np.full(len(train_df), -1, dtype=int)
    if scheme == "grouped":
        for k, (_, va) in enumerate(grouped_folds(train_df, n_folds=n_folds, seed=seed)):
            fold_id[va] = k
    elif scheme == "per_property":
        from sklearn.model_selection import KFold
        for t in TARGETS:
            rows = np.where((train_df["target_type"] == t).values)[0]
            if len(rows) == 0:
                continue
            k_eff = max(2, min(folds_for_target(len(rows), n_folds), len(rows)))
            for k, (_, va) in enumerate(
                    KFold(k_eff, shuffle=True, random_state=seed).split(rows)):
                fold_id[rows[va]] = k
    else:
        raise ValueError(f"unknown fold scheme {scheme!r}")
    assert (fold_id >= 0).all(), "every row must land in exactly one fold"
    return fold_id


# --------------------------------------------------------------------------- #
# base models                                                                  #
# --------------------------------------------------------------------------- #


def physics_feature_cols(target, block, raw_only=False):
    """The fitted physics estimate as an explicit FEATURE, plus a validity flag.

    Partner features alone are not enough. A tree can see `true_egc` and
    `true_eea` but cannot cheaply represent `ei = egc + eea` -- axis-aligned
    splits approximate a sum badly. Handing the model the computed relation as
    one column lets it learn *when to trust it* per row, which is strictly more
    expressive than a single global blend weight applied afterwards.

    Returns (raw_estimate, all_partners_measured) or (None, None) when the
    target has no relation. The caller calibrates a*x+b on training rows.
    """
    rel = RELATIONS.get(target)
    if rel is None:
        return None, None
    srcs, expr = rel
    cols = [f"true_{s}" for s in srcs]
    flags = [f"has_{s}" for s in srcs]
    if not all(c in block.columns for c in cols):
        return None, None
    raw = expr(block[cols].values.astype(float))
    have = block[flags].values.astype(bool).all(axis=1) if all(
        f in block.columns for f in flags) else np.ones(len(block), bool)
    return np.asarray(raw, dtype=float), have


def per_property_oof(kind_maker, kind: str, train_df, X_tr, test_df, X_te,
                     fold_id, all_canon, X_all, seed=SEED, n_jobs=None,
                     use_partners=True, use_physics_feature=True,
                     use_partner_ridge=True):
    """OOF + test + whole-universe predictions for one per-property estimator.

    `universe` maps every canonical polymer to a predicted value for each target.
    Stage 2 of the physics blend needs it, because it must estimate a partner
    property for polymers that were never measured for it.

    Partner features (the measured values of a polymer's OTHER properties) are
    the strongest signal available for the DFT block: 88-99% of test rows in
    those targets have their polymer present in train under some property. They
    are built per fold from TRAINING rows only, and `drop_leaky` removes the
    column that would be the row's own label.
    """
    pass
    from sklearn.linear_model import RidgeCV
    _ALPHAS = [0.1, 1.0, 10.0, 100.0]

    def _ridge_feature(kept_tr, y_tr, kept_others):
        """A linear read of the partner block, handed to the model as one column.

        The trees already see every `true_<prop>` column, but an axis-aligned
        split approximates a linear combination of them badly, and the DFT block
        is close to linear in exactly that way. Fitting the combination on the
        training fold and passing the single fitted value turns something the
        tree cannot represent into something it can split on.

        MEASURED on lgbm alone: eps 0.784 -> 0.853, nc 0.849 -> 0.914,
        ei 0.806 -> 0.870. This is the largest single feature-level gain in the
        pipeline, and it lands entirely on the three weakest targets.
        """
        Xr = np.nan_to_num(kept_tr.values.astype(np.float64))
        if len(Xr) < 30 or Xr.shape[1] == 0:
            return None
        mdl = RidgeCV(alphas=_ALPHAS).fit(Xr, y_tr)
        return [mdl.predict(np.nan_to_num(k.values.astype(np.float64))).reshape(-1, 1)
                for k in kept_others]

    oof = np.full(len(train_df), np.nan)
    test_pred = np.full(len(test_df), np.nan)
    universe = {t: np.full(len(all_canon), np.nan) for t in TARGETS}
    uni_df = pd.DataFrame({"canon": all_canon})

    for t in TARGETS:
        rows = np.where((train_df["target_type"] == t).values)[0]
        if len(rows) < 10:
            continue
        y_all = train_df["target"].values
        f = fold_id[rows]

        for k in np.unique(f):
            tr = rows[f != k]
            va = rows[f == k]
            if len(tr) < 5:
                continue
            if use_partners:
                # partner source = everything except the rows being predicted
                keep = np.ones(len(train_df), dtype=bool)
                keep[va] = False
                src = train_df[keep]
                (btr, bva), _ = partners_build(src, [train_df.iloc[tr], train_df.iloc[va]])
                ktr, kva = partners_drop_leaky(btr, t), partners_drop_leaky(bva, t)
                Xa = np.hstack([X_tr[tr], ktr.values])
                Xb = np.hstack([X_tr[va], kva.values])
                if use_partner_ridge:
                    rf = _ridge_feature(ktr, y_all[tr], [ktr, kva])
                    if rf is not None:
                        Xa = np.hstack([Xa, rf[0]])
                        Xb = np.hstack([Xb, rf[1]])
                if use_physics_feature:
                    ra, ha = physics_feature_cols(t, btr)
                    rb, hb = physics_feature_cols(t, bva)
                    if ra is not None:
                        ok = np.isfinite(ra) & ha
                        if ok.sum() >= 15 and np.std(ra[ok]) > 1e-12:
                            a_, b_ = np.polyfit(ra[ok], y_all[tr][ok], 1)
                            Xa = np.hstack([Xa, np.column_stack(
                                [a_ * ra + b_, ha.astype(np.float32)])])
                            Xb = np.hstack([Xb, np.column_stack(
                                [a_ * rb + b_, hb.astype(np.float32)])])
            else:
                Xa, Xb = X_tr[tr], X_tr[va]
            m = kind_maker(kind, t, len(tr), seed)
            m.fit(Xa, y_all[tr])
            oof[va] = m.predict(Xb)

        # full fit: partner source is all of train, guard still applied
        if use_partners:
            (bfull, bte, buni), _ = partners_build(
                train_df, [train_df.iloc[rows], test_df, uni_df])
            kfull = partners_drop_leaky(bfull, t)
            kte, kuni = partners_drop_leaky(bte, t), partners_drop_leaky(buni, t)
            Xfull = np.hstack([X_tr[rows], kfull.values])
            Xtest = np.hstack([X_te, kte.values])
            Xuni = np.hstack([X_all, kuni.values])
            if use_partner_ridge:
                rf = _ridge_feature(kfull, y_all[rows], [kfull, kte, kuni])
                if rf is not None:
                    Xfull = np.hstack([Xfull, rf[0]])
                    Xtest = np.hstack([Xtest, rf[1]])
                    Xuni = np.hstack([Xuni, rf[2]])
            if use_physics_feature:
                rf, hf = physics_feature_cols(t, bfull)
                rt, ht = physics_feature_cols(t, bte)
                ru, hu = physics_feature_cols(t, buni)
                if rf is not None:
                    ok = np.isfinite(rf) & hf
                    if ok.sum() >= 15 and np.std(rf[ok]) > 1e-12:
                        a_, b_ = np.polyfit(rf[ok], y_all[rows][ok], 1)
                        Xfull = np.hstack([Xfull, np.column_stack(
                            [a_ * rf + b_, hf.astype(np.float32)])])
                        Xtest = np.hstack([Xtest, np.column_stack(
                            [a_ * rt + b_, ht.astype(np.float32)])])
                        Xuni = np.hstack([Xuni, np.column_stack(
                            [a_ * ru + b_, hu.astype(np.float32)])])
        else:
            Xfull, Xtest, Xuni = X_tr[rows], X_te, X_all
        full = kind_maker(kind, t, len(rows), seed)
        full.fit(Xfull, y_all[rows])
        te = np.where((test_df["target_type"] == t).values)[0]
        if len(te):
            test_pred[te] = full.predict(Xtest[te])
        universe[t] = full.predict(Xuni)

    return oof, test_pred, universe


# --------------------------------------------------------------------------- #
# stacking                                                                     #
# --------------------------------------------------------------------------- #

def _fit_ridge(Z, y, alphas=STACK_ALPHAS):
    """Pick alpha by a small internal CV on the meta-features."""
    from sklearn.model_selection import KFold
    best, best_a = -np.inf, alphas[0]
    if len(y) >= 20:
        for a in alphas:
            p = np.zeros(len(y))
            for tr, va in KFold(5, shuffle=True, random_state=0).split(Z):
                p[va] = Ridge(alpha=a).fit(Z[tr], y[tr]).predict(Z[va])
            s = r2(y, p)
            if s > best:
                best, best_a = s, a
    return Ridge(alpha=best_a).fit(Z, y), best_a


def stack(train_df, base_oof: dict, base_test: dict, test_df, fold_id,
          mode: str = "auto"):
    """Combine the base models, per target.

    MEASURED: with three correlated gradient-boosting models, a plain mean beats
    a cross-fitted Ridge meta-learner (0.8961 vs 0.8950 after physics), and Ridge
    over a *single* base model is worse than that model alone (0.8622 vs 0.8645).
    A meta-learner has to earn its variance, and on a 220-row target with inputs
    that correlate at ~0.99 it does not.

    mode="auto" therefore starts from the mean and only tries the cross-fitted
    Ridge once there are at least three base models, keeping it per target only
    where it actually beats the mean out-of-fold. Measured on lgbm+xgb+cb it
    selects ridge for tg, egc and eea and the mean for the rest, worth +0.0004.
    """
    names = sorted(base_oof)
    stacked_oof = np.full(len(train_df), np.nan)
    stacked_test = np.full(len(test_df), np.nan)
    chosen = {}

    use_ridge_allowed = (mode == "ridge") or (mode == "auto" and len(names) >= 3)

    for t in TARGETS:
        rows = np.where((train_df["target_type"] == t).values)[0]
        if len(rows) == 0:
            continue
        te = np.where((test_df["target_type"] == t).values)[0]
        Z = np.nan_to_num(np.column_stack([base_oof[n][rows] for n in names]))
        Zt = np.nan_to_num(np.column_stack([base_test[n][te] for n in names])) \
            if len(te) else np.zeros((0, len(names)))
        y = train_df["target"].values[rows]
        f = fold_id[rows]

        mean_oof = Z.mean(axis=1)
        pick = "mean"

        if use_ridge_allowed and len(rows) >= 60:
            Zx = np.column_stack([Z, Z.std(axis=1)])
            ridge_oof = np.full(len(rows), np.nan)
            for k in np.unique(f):
                tr_m, va_m = f != k, f == k
                if tr_m.sum() < 20 or va_m.sum() == 0:
                    continue
                mdl, _ = _fit_ridge(Zx[tr_m], y[tr_m])
                ridge_oof[va_m] = mdl.predict(Zx[va_m])
            if np.isfinite(ridge_oof).all() and r2(y, ridge_oof) > r2(y, mean_oof):
                pick = "ridge"
                stacked_oof[rows] = ridge_oof
                if len(te):
                    full, _ = _fit_ridge(Zx, y)
                    Ztx = np.column_stack([Zt, Zt.std(axis=1)])
                    stacked_test[te] = full.predict(Ztx)

        if pick == "mean":
            stacked_oof[rows] = mean_oof
            if len(te):
                stacked_test[te] = Zt.mean(axis=1)
        chosen[t] = pick

    return stacked_oof, stacked_test, chosen


# --------------------------------------------------------------------------- #
# two-stage physics                                                            #
# --------------------------------------------------------------------------- #

def partner_frame(train_df, all_canon, universe_by_target):
    """canon -> partner value per property, TRUE where measured else predicted.

    Stage 1 of the blend uses the measured value; stage 2 falls back to the
    model's prediction, which is what extends coverage from ~35-62% of rows to
    100%. `universe_by_target[t]` must be a prediction for every canonical
    polymer from a model that never saw that polymer's label for t -- polymers
    without a t label were never in that model's training set, so a full fit is
    legitimately out-of-sample for them.
    """
    idx = {c: i for i, c in enumerate(all_canon)}
    true_w = train_df.pivot_table(index="canon", columns="target_type",
                                  values="target", aggfunc="mean")
    out = pd.DataFrame(index=all_canon)
    is_true = pd.DataFrame(index=all_canon)
    for t in TARGETS:
        pred = np.asarray(universe_by_target[t], dtype=float)
        col = pd.Series(pred, index=all_canon)
        if t in true_w.columns:
            tv = true_w[t].reindex(all_canon)
            have = tv.notna().values
            col = col.where(~have, tv)
            is_true[t] = have
        else:
            is_true[t] = False
        out[t] = col.values
    return out, is_true


def refine_universe(partners, is_true, train_df, n_rounds=3, damp=0.5,
                    verbose=False):
    """Belief propagation over the property graph, on the predicted entries only.

    The staged blend showed that even rows with NO measured partner take a large
    physics weight (0.63-0.77), i.e. the relation applied to *predicted* partners
    beats the direct model. If a predicted partner is that useful, it is worth
    improving before it is used: the seven properties form a small constraint
    graph (ei = egc+eea, eea = ei-egc, egb ~ egc, eps ~ nc^2, nc ~ sqrt(eps)) and
    one pass of the model's predictions does not satisfy it.

    Each round re-estimates every predicted cell from its neighbours and damps
    toward it. MEASURED cells are never overwritten -- they are the boundary
    conditions the propagation is anchored on. eps and nc are mutual inverses, so
    damping is required; undamped updates oscillate.
    """
    P = partners.copy()
    for rnd in range(n_rounds):
        updates = {}
        for t, (srcs, expr) in RELATIONS.items():
            if t not in P.columns or any(c not in P.columns for c in srcs):
                continue
            # calibrate on train polymers where t IS measured
            meas = train_df[train_df["target_type"] == t]
            idx = meas["canon"].values
            src = P.reindex(idx)[srcs].values
            ok = np.isfinite(src).all(axis=1)
            if ok.sum() < 25:
                continue
            x = expr(src[ok])
            y = meas["target"].values[ok]
            if not np.isfinite(x).all() or np.std(x) < 1e-12:
                continue
            a, b = np.polyfit(x, y, 1)
            est = a * expr(P[srcs].values) + b
            cur = P[t].values.astype(float)
            upd = np.where(np.isfinite(est), (1 - damp) * cur + damp * est, cur)
            updates[t] = np.where(is_true[t].values, cur, upd)
        for t, v in updates.items():
            P[t] = v
        if verbose:
            print(f"  refine round {rnd+1}: updated {list(updates)}")
    return P


def apply_physics(train_df, oof, test_df, test_pred, fold_id, partners, is_true,
                  oof_by_target=None, n_rounds=3, damp=0.5, verbose=True):
    """Cross-fitted, staged, iterated physics blend.

    Three things are happening, and the third is where the leak lives if you are
    careless:

    1. STAGING. Rows are grouped by how many of the relation's sources are
       actually measured, and each group gets its own calibration and its own
       blend weight. A binary true/predicted split wastes the partially covered
       rows, and they are numerous -- of 148 `ei` test rows, 55 have both sources
       measured, 69 have exactly one.

    2. ITERATION. The seven properties form a constraint graph
       (ei = egc+eea, eea = ei-egc, egb ~ egc, eps ~ nc^2, nc ~ sqrt(eps)) that
       one pass of model predictions does not satisfy, so predicted cells are
       re-estimated from their neighbours and damped toward the estimate.

    3. THE MASK, which makes 1 and 2 honest. `eps` is refined FROM `nc`, and `nc`
       is then predicted FROM the refined `eps`. For a validation polymer whose
       `nc` is measured, its own label would flow into its own prediction -- a
       closed loop that reads as a huge gain and is entirely fake (it took the
       measured score from 0.893 to 0.935 before this mask existed). So for every
       (target, fold) the refinement is redone with that fold's true `target`
       values replaced by their out-of-fold predictions.

       The test side needs no such mask: only 2 of 4940 test rows have their
       (polymer, target_type) present in train, so a test row's own label is not
       in the table to leak.
    """
    oof = np.asarray(oof, dtype=float)
    oof_out = oof.copy()
    test_out = np.asarray(test_pred, dtype=float).copy()
    info = {}

    canon_pos = {c: i for i, c in enumerate(partners.index)}

    for t, (srcs, expr) in RELATIONS.items():
        rows = np.where((train_df["target_type"] == t).values)[0]
        if len(rows) < 30:
            continue
        y = train_df["target"].values[rows]
        canon = train_df["canon"].values[rows]
        f = fold_id[rows]

        blended = oof_out[rows].copy()
        weights, levels = {}, np.zeros(len(rows), dtype=int)

        for k in np.unique(f):
            va = f == k
            if va.sum() == 0:
                continue
            # --- mask: hide this fold's own target labels, everywhere ---
            Pk = partners.copy()
            Ik = is_true.copy()
            pairs = [(canon_pos[c], v) for c, v in
                     zip(canon[va], oof[rows[va]]) if c in canon_pos]
            if pairs:
                pos = [i for i, _ in pairs]
                col = Pk[t].values.astype(float)
                col[pos] = [v for _, v in pairs]        # out-of-fold stand-in
                Pk[t] = col
                flag = Ik[t].values.copy()
                flag[pos] = False
                Ik[t] = flag
            Pk = refine_universe(Pk, Ik, train_df.iloc[np.setdiff1d(
                np.arange(len(train_df)), rows[va])], n_rounds=n_rounds, damp=damp)

            n_true_k = Ik.reindex(canon)[srcs].sum(axis=1).values.astype(int)
            levels[va] = n_true_k[va]
            for lvl in sorted(set(n_true_k.tolist()), reverse=True):
                tr_m = (n_true_k == lvl) & (~va)
                va_m = (n_true_k == lvl) & va
                if tr_m.sum() < 15 or va_m.sum() == 0:
                    continue
                rel = _fit_relation(train_df, rows[tr_m], t, srcs, expr, Pk)
                if rel is None:
                    continue
                e_tr = rel.apply(Pk.reindex(canon[tr_m])[srcs].values)
                e_va = rel.apply(Pk.reindex(canon[va_m])[srcs].values)
                w, _ = tune_weight(y[tr_m], oof[rows[tr_m]], e_tr)
                blended[va_m] = blend(oof[rows[va_m]], e_va, w)
                weights.setdefault(lvl, []).append(w)

        gain = r2(y, blended) - r2(y, oof[rows])
        info[t] = {"r2_before": r2(y, oof[rows]), "r2_after": r2(y, blended),
                   "gain": gain,
                   "n_by_level": {int(l): int((levels == l).sum())
                                  for l in sorted(set(levels.tolist()), reverse=True)},
                   "w_by_level": {int(l): round(float(np.mean(v)), 3)
                                  for l, v in sorted(weights.items(), reverse=True)}}
        if gain > 0:
            oof_out[rows] = blended

        # ---- test side: no mask needed, calibration on all train rows ----
        te = np.where((test_df["target_type"] == t).values)[0]
        if len(te) and gain > 0:
            Pfull = refine_universe(partners, is_true, train_df,
                                    n_rounds=n_rounds, damp=damp)
            tcanon = test_df["canon"].values[te]
            n_true_tr = is_true.reindex(canon)[srcs].sum(axis=1).values.astype(int)
            t_lvl = is_true.reindex(tcanon)[srcs].sum(axis=1).values.astype(int)
            for lvl in sorted(set(n_true_tr.tolist()), reverse=True):
                tr_m = n_true_tr == lvl
                te_m = t_lvl == lvl
                if tr_m.sum() < 15 or te_m.sum() == 0:
                    continue
                rel = _fit_relation(train_df, rows[tr_m], t, srcs, expr, Pfull)
                if rel is None:
                    continue
                e_tr = rel.apply(Pfull.reindex(canon[tr_m])[srcs].values)
                w, _ = tune_weight(y[tr_m], oof[rows[tr_m]], e_tr)
                e_te = rel.apply(Pfull.reindex(tcanon[te_m])[srcs].values)
                test_out[te[te_m]] = blend(test_out[te[te_m]], e_te, w)

    if verbose and info:
        print(f"\n{'target':<6}{'before':>9}{'after':>9}{'gain':>9}   "
              f"{'rows by #measured sources':<26}{'blend weight by level'}")
        for t, d in info.items():
            print(f"{t:<6}{d['r2_before']:>9.4f}{d['r2_after']:>9.4f}{d['gain']:>+9.4f}   "
                  f"{str(d['n_by_level']):<26}{d['w_by_level']}")
    return oof_out, test_out, info


def _fit_relation(train_df, row_idx, target, srcs, expr, partners):
    """Calibrate a*x+b on the given TRAINING rows only."""
    canon = train_df["canon"].values[row_idx]
    y = train_df["target"].values[row_idx]
    src = partners.reindex(canon)[srcs].values
    ok = np.isfinite(src).all(axis=1) & np.isfinite(y)
    if ok.sum() < 15:
        return None
    x = expr(src[ok])
    if not np.isfinite(x).all() or np.std(x) < 1e-12:
        return None
    a, b = np.polyfit(x, y[ok], 1)
    return Relation(target, list(srcs), expr, float(a), float(b), int(ok.sum()))


def partner_regression(train_df, pred, test_df, test_pred, fold_id, partners,
                       is_true, verbose=True):
    """Generalise the hand-written relations to a learned partner combination.

    `RELATIONS` encodes the five relations a chemist can write down:
    ei = egc+eea, egb ~ egc, eps ~ nc^2 and so on. But a polymer's other measured
    properties carry more signal than those five expressions extract -- the whole
    DFT block is one electronic-structure description, so `eps` is informative
    about `nc` *and* about `ei`, not only through the Maxwell relation.

    So per target: ridge-regress the target on ALL other properties' values plus
    a measured/predicted flag for each, cross-fitted over the folds, and blend the
    result in with a cross-fitted weight. This subsumes the hand relations rather
    than replacing them -- it runs after them, on their output.

    MEASURED, on the four-model stack: +0.0022 mean, concentrated exactly where
    the score gap is -- eps +0.0077, nc +0.0067, egc +0.0012, everything else
    flat. The estimate itself reaches R2 0.84 on eps and 0.90 on nc from partner
    values alone, with no molecular features at all.

    No cycle, so no mask is needed: the target's own column is excluded from the
    design matrix, and the predicted entries in the other columns come from models
    that never saw this row's label for this target.
    """
    from sklearn.linear_model import RidgeCV

    out = np.asarray(pred, dtype=float).copy()
    test_out = np.asarray(test_pred, dtype=float).copy()
    info = {}
    alphas = [0.1, 1.0, 10.0, 100.0]

    for t in TARGETS:
        rows = np.where((train_df["target_type"] == t).values)[0]
        if len(rows) < 60:
            continue
        srcs = [c for c in TARGETS if c != t]
        canon = train_df["canon"].values[rows]
        y = train_df["target"].values[rows]
        f = fold_id[rows]

        def design(idx_canon):
            return np.nan_to_num(np.column_stack([
                partners.reindex(idx_canon)[srcs].values,
                is_true.reindex(idx_canon)[srcs].values.astype(float)]))

        Xp = design(canon)
        est = np.full(len(rows), np.nan)
        for k in np.unique(f):
            a, b = f != k, f == k
            if a.sum() < 30 or b.sum() == 0:
                continue
            est[b] = RidgeCV(alphas=alphas).fit(Xp[a], y[a]).predict(Xp[b])
        ok = np.isfinite(est)
        if ok.sum() < 30:
            continue

        blended = out[rows].copy()
        for k in np.unique(f):
            a, b = (f != k) & ok, (f == k) & ok
            if a.sum() < 30 or b.sum() == 0:
                continue
            w, _ = tune_weight(y[a], out[rows[a]], est[a])
            blended[b] = blend(out[rows[b]], est[b], w)

        gain = r2(y, blended) - r2(y, out[rows])
        info[t] = {"est_r2": r2(y[ok], est[ok]), "gain": gain}
        if gain <= 0:
            continue
        out[rows] = blended

        te = np.where((test_df["target_type"] == t).values)[0]
        if len(te):
            mdl = RidgeCV(alphas=alphas).fit(Xp[ok], y[ok])
            e_tr = mdl.predict(Xp)
            w, _ = tune_weight(y, np.asarray(pred, dtype=float)[rows], e_tr)
            e_te = mdl.predict(design(test_df["canon"].values[te]))
            test_out[te] = blend(test_out[te], e_te, w)

    if verbose and info:
        print(f"\n{'target':<7}{'partner-ridge R2':>18}{'blend gain':>13}")
        for t, dct in info.items():
            print(f"{t:<7}{dct['est_r2']:>18.4f}{dct['gain']:>+13.4f}")
    return out, test_out, info


def report(train_df, oof, label=""):
    df = train_df.copy()
    df["pred"] = oof
    score, per = competition_score(df)
    print(f"\n=== {label} ===")
    for t in TARGETS:
        print(f"  {t:<4} R2 = {per[t]:+.4f}")
    print(f"  {'MEAN':<4}    = {score:+.4f}")
    return score, per


## 11. Run the pipeline and write `submission.csv`


In [ ]:
MODELS = ['lgbm', 'xgb', 'cb', 'mtnn']
t_start = time.time()

all_canon = list(dict.fromkeys(list(train_df["canon"]) + list(test_df["canon"])))
print(f"universe: {len(all_canon)} distinct polymers")

X_tr = featurize(train_df["canon"])
X_te = featurize(test_df["canon"])
X_all = featurize(all_canon)
print(f"features {X_tr.shape}  ({time.time()-t_start:.0f}s)")

fold_id = build_fold_id(train_df, n_folds=N_FOLDS, seed=SEED)

base_oof, base_test, tree_universe = {}, {}, {}
for name in MODELS:
    t1 = time.time()
    if name in ("lgbm", "xgb", "cb"):
        o, te, uni = per_property_oof(make, name, train_df, X_tr, test_df, X_te,
                                      fold_id, all_canon, X_all, seed=SEED)
        tree_universe[name] = np.column_stack([uni[t] for t in TARGETS])
    elif name == "mtnn":
        o, te = mtnn_oof_and_test(train_df, X_tr, test_df, X_te, fold_id, NN_SEEDS)
    else:
        raise ValueError(name)
    base_oof[name], base_test[name] = np.asarray(o), np.asarray(te)
    _d = train_df.copy(); _d["pred"] = base_oof[name]
    print(f"  {name:<6} mean R2 {competition_score(_d)[0]:+.4f}   ({time.time()-t1:.0f}s)")

print(f"\n{'model':<8}" + "".join(f"{t:>8}" for t in TARGETS) + f"{'MEAN':>9}")
for name in MODELS:
    _d = train_df.copy(); _d["pred"] = base_oof[name]
    s, per = competition_score(_d)
    print(f"{name:<8}" + "".join(f"{per[t]:>8.4f}" for t in TARGETS) + f"{s:>9.4f}")

stacked_oof, stacked_test, alphas = stack(train_df, base_oof, base_test, test_df, fold_id)
score_stack, _ = report(train_df, stacked_oof, "stacked")

uni = np.mean([tree_universe[k] for k in tree_universe], axis=0)
universe_by_target = {t: uni[:, i] for i, t in enumerate(TARGETS)}
partners, is_true = partner_frame(train_df, all_canon, universe_by_target)
# n_rounds=2 is what was measured locally. The refinement itself is worth only
# ~+0.001 once the per-fold mask is in place -- the mask is the important part.
# n_rounds=0: the relation-graph refinement measured slightly NEGATIVE on the
# four-model stack once its per-fold mask was in place (0.9047 vs 0.9040). The
# masking logic is kept in the code because it documents why the unmasked version
# was fake, but it ships disabled.
final_oof, final_test, phys_info = apply_physics(
    train_df, stacked_oof, test_df, stacked_test, fold_id, partners, is_true,
    n_rounds=0)
report(train_df, final_oof, "stacked + staged physics")

final_oof, final_test, pr_info = partner_regression(
    train_df, final_oof, test_df, final_test, fold_id, partners, is_true)
FINAL_SCORE, FINAL_PER = report(train_df, final_oof, "+ generalized partner regression")

# clip to the observed range: a negative bandgap is not a polymer, and one wild
# extrapolation can wreck an R2 computed over only ~150 test rows
final = np.asarray(final_test, dtype=float)
for t in TARGETS:
    m = (test_df["target_type"] == t).values
    if not m.any():
        continue
    v = train_df.loc[train_df.target_type == t, "target"]
    lo, hi = v.min(), v.max(); pad = 0.05 * (hi - lo)
    n_clip = int(((final[m] < lo - pad) | (final[m] > hi + pad)).sum())
    final[m] = np.clip(final[m], lo - pad, hi + pad)
    if n_clip:
        print(f"  clipped {n_clip}/{m.sum()} {t} predictions")
bad = ~np.isfinite(final)
if bad.any():
    for t in TARGETS:
        m = (test_df["target_type"] == t).values & bad
        if m.any():
            final[m] = float(train_df.loc[train_df.target_type == t, "target"].mean())
    print(f"replaced {int(bad.sum())} non-finite predictions with the per-target mean")

submission = pd.DataFrame({"id": test_df["id"].values, "target": final})
submission.to_csv(os.path.join(WORK_DIR, "submission.csv"), index=False)
print(f"\nwrote submission.csv: {len(submission)} rows   total {time.time()-t_start:.0f}s")
print(submission.head().to_string(index=False))


## 12. Explainability — per-target TreeSHAP

Exact SHAP values from LightGBM's own `pred_contrib=True`: the same algorithm the `shap` package implements, computed inside LightGBM, so the notebook gains no dependency that could fail to install.


In [ ]:
FAMILY = {"rd": "RDKit descriptors", "mfp2": "Morgan r=2", "mfp3": "Morgan r=3",
          "ap": "atom pair", "tt": "topological torsion", "mac": "MACCS keys",
          "po": "polymer-specific", "grp": "functional groups"}
_cols = feature_names()

for t in TARGETS:
    rows = np.where((train_df["target_type"] == t).values)[0]
    if len(rows) < 20:
        continue
    idx = rows if len(rows) <= 800 else np.random.RandomState(SEED).choice(rows, 800, replace=False)
    mdl = make("lgbm", t, len(rows), SEED)
    mdl.fit(X_tr[rows], train_df["target"].values[rows])
    booster = mdl.model.booster_
    contrib = booster.predict(X_tr[idx], pred_contrib=True)
    vals = np.abs(contrib[:, :-1]).mean(axis=0)
    s = pd.Series(vals, index=_cols).sort_values(ascending=False)

    fam = s.groupby([c.split("_", 1)[0] for c in s.index]).sum()
    fam = (fam / fam.sum() * 100).sort_values(ascending=False)
    print(f"\n===== {t}  (n={len(rows)}) =====")
    print("  attribution by feature family:")
    for k, v in fam.items():
        if v >= 0.5:
            print(f"    {FAMILY.get(k, k):<22} {v:5.1f}%")
    interp = s[[c for c in s.index if c.startswith(("rd_", "po_", "grp_"))]]
    print("  top interpretable features:")
    for k, v in interp.head(8).items():
        print(f"    {k:<32} {v:.4g}")


## 13. Polymer-invariance certificate


In [ ]:
# Three ways the same polymer can be written differently. Permutational
# invariance is EXACT by construction here, because every feature is computed
# from canonicalize(), which is idempotent. Translational and repetition
# rewritings change the graph RDKit sees, so those are measured, not claimed.
_rng = np.random.RandomState(SEED)
_idx = _rng.choice(len(test_df), size=min(150, len(test_df)), replace=False)
_sample = test_df.iloc[_idx].reset_index(drop=True)

def _predict_rows(df):
    """Score a frame through the same featurisation the pipeline uses."""
    Xq = featurize(df["canon"])
    out = np.zeros(len(df))
    for t in TARGETS:
        m = (df["target_type"] == t).values
        if not m.any():
            continue
        out[m] = _INV_MODELS[t].predict(Xq[m])
    return out

_INV_MODELS = {}
for t in TARGETS:
    rows = np.where((train_df["target_type"] == t).values)[0]
    mdl = make("lgbm", t, len(rows), SEED)
    mdl.fit(X_tr[rows], train_df["target"].values[rows])
    _INV_MODELS[t] = mdl

_base = _predict_rows(_sample)
print(f"{'rewriting':<18}{'changed':>9}{'canon moved':>13}{'median |d|':>13}{'max |d|':>12}")
for _kind, _fn in [("permutational", lambda s, i: randomize(s, seed=i)),
                   ("translational", lambda s, i: translate(s, k=i + 1)),
                   ("repetition",    lambda s, i: build_oligomer(s, n=i + 2))]:
    _d, _nch, _ncanon = [], 0, 0
    for _i in range(2):
        _v = _sample.copy()
        _v["smiles"] = [_fn(s, _i) for s in _sample["smiles"]]
        _v["canon"] = _v["smiles"].map(canonicalize)
        _ch = (_v["smiles"] != _sample["smiles"]).values
        _nch += int(_ch.sum())
        _ncanon += int((_v["canon"] != _sample["canon"]).values.sum())
        _p = _predict_rows(_v)
        _d.extend(np.abs(_p - _base)[_ch])
    _d = np.array(_d) if _d else np.array([0.0])
    print(f"{_kind:<18}{_nch:>9}{_ncanon:>13}{np.median(_d):>13.6f}{_d.max():>12.4f}")

_v = _sample.copy()
_v["smiles"] = [randomize(s, seed=0) for s in _sample["smiles"]]
_v["canon"] = _v["smiles"].map(canonicalize)
_perm_max = float(np.abs(_predict_rows(_v) - _base).max())
assert _perm_max < 1e-9, f"permutational invariance broken: max delta {_perm_max}"
print(f"\nPASS: permutational invariance is EXACT (max delta {_perm_max:.1e}).")
print("Every feature derives from the canonical SMILES, and canonicalisation is idempotent,")
print("so any re-ordering of the atoms in the input string yields a bit-identical prediction.")


## 14. Compliance self-audit


In [ ]:
import glob as _glob
_fail, _warn = [], []

_sub = pd.read_csv(os.path.join(WORK_DIR, "submission.csv"))
_test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
if list(_sub.columns) != ["id", "target"]:
    _fail.append(f"columns are {list(_sub.columns)}, expected ['id','target']")
if len(_sub) != len(_test):
    _fail.append(f"{len(_sub)} rows, test.csv has {len(_test)}")
if set(_sub["id"]) != set(_test["id"]):
    _fail.append("id set does not match test.csv")
if _sub["id"].duplicated().any():
    _fail.append("duplicate ids")
if not np.isfinite(_sub["target"]).all():
    _fail.append("non-finite predictions")

_m = _test.merge(_sub, on="id")
for _t in TARGETS:
    _v = _m.loc[_m.target_type == _t, "target"]
    _o = train_df.loc[train_df.target_type == _t, "target"]
    if len(_v) and _v.std() < 1e-8:
        _fail.append(f"{_t}: all predictions identical")
    print(f"  {_t:<4} n={len(_v):<5} pred [{_v.min():9.4g}, {_v.max():9.4g}]"
          f"   train [{_o.min():9.4g}, {_o.max():9.4g}]")

_others = [p for p in _glob.glob("/kaggle/input/*")
           if os.path.abspath(p) != os.path.abspath(DATA_DIR)]
if _others:
    _fail.append(f"other datasets attached: {_others} -- rule 6.2.1 requires only competition data")

print()
print(f"data read from : {DATA_DIR}")
print(f"other inputs   : {_others or 'none'}")
print(f"seeds          : SEED={SEED}, NN_SEEDS={NN_SEEDS} (set and printed at the top)")
print(f"artifacts read : none (nothing deserialised from disk; no checkpoint import)")
print(f"wall-clock deps: none (every loop is a fixed fold/epoch count)")
print(f"local OOF score: {FINAL_SCORE:.4f}")
for _w in _warn:
    print("WARN ", _w)
for _f in _fail:
    print("FAIL ", _f)
assert not _fail, f"compliance audit failed: {_fail}"
print("\nPASS: submission.csv is well formed and the run is rule-compliant.")
